Task: KPI Engineering & Feature Extraction

Create military strength KPIs by calculating asset power metrics, financial ratios, rankings, regional classifications, and strategic groupings for comparative analysis and dashboards.

Stage-1: Load clean dataset

Loaded the cleaned dataset(military_cleaned_ready.csv) and standardized country names to ensure consistent mapping and analysis.

In [1]:
import pandas as pd

df = pd.read_csv("military_cleaned_ready.csv")

# clean country names (important)
df["country"] = df["country"].str.strip()

print(df.shape)
df.head()

(145, 38)


,country,total_population,gdp,defense_budget,total_manpower,active_personnel,reserve_personnel,air_force_personnel,army_personnel,navy_personnel,...,external_debt,power_index,tanks_stock,tanks_readiness,self_propelled_artillery_stock,self_propelled_artillery_readiness,towed_artillery_stock,towed_artillery_readiness,rocket_artillery_stock,rocket_artillery_readiness
0,Chile,18664652,596556000000,6800000000,8959033,80000,40000,7645,80000,25000,...,235950000000,0.8704,386,251,48,31,249,162,9,6
1,Dominican Republic,10815857,276884000000,903000000,4975294,89000,189000,6730,28750,13200,...,35044000000,2.5853,6,3,0,0,14,7,0,0
2,Norway,5509733,507680000000,11189483200,2203893,33440,20100,3650,8735,4010,...,787758400000,0.6679,36,25,28,20,0,0,0,0
3,Zambia,20799116,79207000000,385321000,7279691,15150,0,3745,13250,0,...,16597000000,2.4461,75,41,12,7,60,33,50,28
4,Liberia,5437249,9308000000,20700000,2392390,2100,0,0,2100,300,...,1335000000,3.9275,0,0,0,0,0,0,0,0


Stage-2: Create Total Assets & Assets per capita

Combined total_aircraft, tanks_stock, and naval_assets to compute total military assets and derived assets per capita using manpower.

In [162]:
df["total_assets"] = (
    df["total_aircraft"] +
    df["tanks_stock"] +   # use tanks_stock in READY dataset
    df["naval_assets"]
)

df["assets_per_capita"] = df["total_assets"] / df["total_manpower"]
df["assets_per_capita"] = df["assets_per_capita"].round(8)

Stage-3: Budget to GDP ratio

Calculated defense spending burden by dividing defense_budget by gdp to measure national defense priority.

In [164]:
df["budget_to_gdp_ratio"] = df["defense_budget"] / df["gdp"]

Stage-4: Sort by PowerIndex

Sorted countries by power_index to establish an accurate strength ranking order.

In [166]:
df = df.sort_values("power_index").reset_index(drop=True)

Stage-5: Generate Rank Column

Assigned strength rankings to countries based on sorted power index positions.

In [168]:
df["rank"] = df.index + 1

Stage-6:Calculate Rank Gap

Computed rank gap to show each country’s distance from the top global rank.

In [170]:
df["rank_gap"] = df["rank"] - 1

Stage-7:Metadata Enrichment

Installed the pycountry_convert package to enable automatic mapping of countries to continent codes for regional analysis.

In [172]:
pip install pycountry_convert

Note: you may need to restart the kernel to use updated packages.


Converted country names into continent codes and stored them in continent_code to support regional analysis and dashboards.

In [173]:

import pycountry_convert as pc

def get_continent(country):
    try:
        country_code = pc.country_name_to_country_alpha2(country)
        continent = pc.country_alpha2_to_continent_code(country_code)
        return continent
    except:
        return "Unknown"

df["continent_code"] = df["country"].apply(get_continent)


Nato member list creation

Defined NATO member countries to support alliance-based strength analysis.

In [175]:
nato = [
"Albania","Belgium","Bulgaria","Canada","Croatia","Czech Republic",
"Denmark","Estonia","Finland","France","Germany","Greece","Hungary",
"Iceland","Italy","Latvia","Lithuania","Luxembourg","Montenegro",
"Netherlands","North Macedonia","Norway","Poland","Portugal",
"Romania","Slovakia","Slovenia","Spain","Sweden","Turkey",
"United Kingdom","United States"
]

df["nato_member"] = df["country"].isin(nato)

Defence priority category

Classified countries into Low, Moderate, and High defense priority using the budget-to-GDP ratio.

In [176]:
df["defense_priority"] = pd.cut(
    df["budget_to_gdp_ratio"],
    bins=[0,0.02,0.04,1],
    labels=["Low","Moderate","High"]
)

Power tier classification

Grouped countries into Super, Major, Regional, and Emerging powers based on rank ranges.

In [177]:
df["power_tier"] = pd.cut(
    df["rank"],
    bins=[0,10,30,70,200],
    labels=["Super Power","Major Power","Regional Power","Emerging"]
)

Stage-8:KPI summary statistics

Generated descriptive statistics for assets per capita and budget ratio to understand global distribution patterns.

In [178]:
df[["assets_per_capita","budget_to_gdp_ratio"]].describe()


,assets_per_capita,budget_to_gdp_ratio
count,145.000000,145.000000
mean,0.000034,0.019467
std,0.000055,0.029967
min,0.000000,0.000085
25%,0.000004,0.005833
50%,0.000011,0.012138
75%,0.000036,0.023780
max,0.000293,0.308121


Reviewed KPI values for major powers like the United States and India to validate metric accuracy.

In [179]:
df[df["country"]=="United States"]
df[df["country"]=="India"]

,country,total_population,gdp,defense_budget,total_manpower,active_personnel,reserve_personnel,air_force_personnel,army_personnel,navy_personnel,...,assets_per_capita,budget_to_gdp_ratio,rank,rank_gap,continent,nato_member,defense_priority,power_tier,assets_per_million,continent_code
3,India,1409128296,14244000000000,109000000000,662290299,1431000,1000000,289000,2148000,148869,...,5.200000e-07,0.007652,4,3,Asia,False,Low,Super Power,0.52,AS


Stage-9:Validating KPI values

Replaced infinite values and filled missing numeric data with 0 to ensure stable KPI calculations.

In [180]:
import numpy as np

df.replace([np.inf, -np.inf], np.nan, inplace=True)

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(0)

In [189]:
print(df.dtypes)

country                                 object
total_population                         int64
gdp                                      int64
defense_budget                           int64
total_manpower                           int64
active_personnel                         int64
reserve_personnel                        int64
air_force_personnel                      int64
army_personnel                           int64
navy_personnel                           int64
total_aircraft                         float64
attack_aircraft                        float64
fighter_aircraft                       float64
transport_aircraft                     float64
helicopters                            float64
special_mission_aircraft               float64
trainer_aircraft                       float64
attack_helicopters                     float64
tanker_aircraft                        float64
naval_assets                           float64
aircraft_carriers                      float64
helicopter_ca

Checked for countries without continent mapping to prevent regional grouping errors.

In [191]:
df[df["continent"].isna()][["country"]]

,country


Corrected country name inconsistencies (e.g., Korea variants) to improve mapping accuracy.

In [193]:
name_fix = {
    "Beliz": "Belize",
    "Korea, South": "South Korea",
    "Korea, North": "North Korea"
}

df["country"] = df["country"].replace(name_fix)

Manually assigned continents for unmatched countries to ensure complete regional classification.

In [195]:
continent_fix = {
    "South Korea": "Asia",
    "North Korea": "Asia",
    "Democratic Republic Of The Congo": "Africa",
    "Republic Of The Congo": "Africa",
    "Bosnia And Herzegovina": "Europe",
    "Kosovo": "Europe",
    "Belize": "North America"
}

df["continent"] = df["continent"].fillna(df["country"].map(continent_fix))

In [197]:
df[df["continent"].isna()]

,country,total_population,gdp,defense_budget,total_manpower,active_personnel,reserve_personnel,air_force_personnel,army_personnel,navy_personnel,...,assets_per_capita,budget_to_gdp_ratio,rank,rank_gap,continent,nato_member,defense_priority,power_tier,assets_per_million,continent_code



Scaled assets per capita into assets per million people for easier comparison.


In [201]:
df["assets_per_million"] = (
    df["assets_per_capita"] * 1_000_000
).round(2)

Added Region using Country Converter:

Enhanced dataset with continent/region classification for geographic analysis.

In [211]:
%pip install country_converter

In [219]:
%pip install --upgrade country_converter

In [227]:
import pandas as pd
import country_converter as coco

# load dataset
df = pd.read_csv("military_final9.csv")

cc = coco.CountryConverter()

# continent (works in all versions)
df['continent'] = cc.convert(names=df['country'], to='continent')

# ===============================
# CREATE SUBREGION FROM CONTINENT + COUNTRY LOGIC
# ===============================

middle_east = [
    'Saudi Arabia','Iran','Iraq','Israel','Jordan','Kuwait','Qatar','UAE',
    'Oman','Yemen','Syria','Lebanon','Bahrain','Turkey'
]

south_asia = ['India','Pakistan','Bangladesh','Sri Lanka','Nepal','Bhutan','Maldives','Afghanistan']

east_asia = ['China','Japan','South Korea','North Korea','Mongolia','Taiwan']

southeast_asia = ['Indonesia','Malaysia','Singapore','Thailand','Vietnam',
                  'Philippines','Myanmar','Cambodia','Laos','Brunei']

western_europe = ['United Kingdom','France','Germany','Netherlands','Belgium','Switzerland','Austria','Ireland','Luxembourg']

eastern_europe = ['Russia','Ukraine','Poland','Romania','Hungary','Czech Republic','Slovakia','Bulgaria','Belarus']

north_america = ['United States','Canada','Mexico']

south_america = ['Brazil','Argentina','Chile','Colombia','Peru','Venezuela','Ecuador','Bolivia','Paraguay','Uruguay']

def get_subregion(country, continent):
    
    if country in middle_east:
        return "Middle East"
    elif country in south_asia:
        return "South Asia"
    elif country in east_asia:
        return "East Asia"
    elif country in southeast_asia:
        return "Southeast Asia"
    elif country in western_europe:
        return "Western Europe"
    elif country in eastern_europe:
        return "Eastern Europe"
    elif country in north_america:
        return "North America"
    elif country in south_america:
        return "South America"
    else:
        return continent   # fallback

df['region'] = df.apply(lambda x: get_subregion(x['country'], x['continent']), axis=1)

# replace missing continent values
df['continent'] = df['continent'].replace('not found', 'Other')

print(df[['country','continent','region']].head())

df.to_csv("military_with_regions.csv", index=False)

print("✅ Regions added successfully!")

              country continent         region
0       United States   America  North America
1  Russian Federation    Europe         Europe
2               China      Asia      East Asia
3               India      Asia     South Asia
4         South Korea      Asia      East Asia
✅ Regions added successfully!


Tagged countries as NATO or Non-NATO to support alliance strength comparisons.

In [229]:
df["nato_member"] = df["country"].apply(lambda x: "NATO" if x in nato else "Non-NATO")

Stage-9: Saved final dataset:

Saved the final enriched dataset(military_with_regions2.csv) with KPIs, rankings, regions, and alliance labels ready for dashboard visualization.

In [231]:
df.to_csv("military_with_regions2.csv", index=False)

print("✅ Regions added successfully!")

✅ Regions added successfully!


In [263]:
pip install pycountry

In [267]:
import pandas as pd
import pycountry

# load dataset
df = pd.read_csv("military_with_regions2.csv")

# function to get ISO country code
def get_country_code(name):
    try:
        return pycountry.countries.lookup(name).alpha_2.lower()
    except:
        return None

# create code column
df["code"] = df["country"].apply(get_country_code)

# create flag URL column
df["flag_url"] = df["code"].apply(
    lambda x: f"https://flagcdn.com/w40/{x}.png" if x else None
)

# save new file
df.to_csv("military_with_flags.csv", index=False)

print("Flags added successfully!")

Flags added successfully!
